# Data Cleaning
This is my inital notebook for data cleaning. I am going to use the 2025 data, and I want to re-map all the numeric values from the survey to their true values in the codebook

In [37]:
library(tidyverse)
library(readxl)

npors_2025 <- read_csv('../data/rawdata/NPORS_2025_for_public_release_FINAL.csv')
codebook_2025 <- read_excel("../data/rawdata/codebooks/NPORS_2025_codebook_FINAL.xlsx")


Rows: 5022 Columns: 65
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (63): RESPID, MODE, LANGUAGE, LANGUAGEINITIAL, STRATUM, ECON1MOD, ECON1...
date  (2): INTERVIEW_START, INTERVIEW_END

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [38]:
#clean codebook to get rid of continuous lables
codebook_2025 <- codebook_2025 |>
  filter(!str_detect(Value_Labels, "Min|Max"))

In [43]:
#making a lookup table
lookups_2025 <- codebook_2025 |>
  group_by(Variable) |>
  summarise(map = list(setNames(Value_Labels, Values))) |>
  deframe()

#example
lookups_2025$UNITY


                                                                 1 
 "Americans are united when it comes to the most important values" 
                                                                 2 
"Americans are divided when it comes to the most important values" 
                                                                99 
                                               "Refused/Web blank" 

In [6]:
#applying the labels as factors
npors_2025_labeled <- npors_2025 |>
  mutate(across(
    names(lookups),
    ~ factor(., levels = names(lookups[[cur_column()]]),
             labels = lookups[[cur_column()]])
  ))

In [7]:
# dropping irrelevant columns in the data
npors_2025_labeled <- npors_2025_labeled |> select(-MODE, -LANGUAGE, -LANGUAGEINITIAL, -INTERVIEW_START,
-INTERVIEW_END, -INTFREQ, -RACECMB, -HISP)

In [8]:
head(npors_2025_labeled)

# A tibble: 6 × 57
  RESPID STRATUM           ECON1MOD ECON1BMOD COMTYPE2 UNITY CRIMESAFE GOVPROTCT
   <dbl> <fct>             <fct>    <fct>     <fct>    <fct> <fct>     <fct>    
1   1470 10 = Remaining C… Poor     Worse     Rural    Amer… Very safe Sometime…
2   2374 7 = 75%+ Hispani… Only fa… Worse     Urban    Amer… Somewhat… It's not…
3   1177 5 = 50%-74.99% H… Good     Better    Rural    Amer… Very safe It's not…
4  15459 10 = Remaining C… Only fa… About th… Rural    Amer… Very safe Sometime…
5   9849 9 = Remaining CB… Good     Better    Suburban Amer… Somewhat… It's not…
6   8178 10 = Remaining C… Good     Worse     Urban    Amer… Very safe It's not…
# ℹ 49 more variables: MOREGUNIMPACT <fct>, FIN_SIT <fct>, VET1 <fct>,
#   VOL12_CPS <fct>, EMINUSE <fct>, INTMOB <fct>, INTFREQ_COLLAPSED <fct>,
#   HOME4NW2 <fct>, BBHOME <fct>, SMUSE_FB <fct>, SMUSE_YT <fct>,
#   SMUSE_X <fct>, SMUSE_IG <fct>, SMUSE_SC <fct>, SMUSE_WA <fct>,
#   SMUSE_TT <fct>, SMUSE_RD <fct>, SMUSE_BSK <fct>, S

In [9]:
#now to check for missing values
na_counts <- npors_2025_labeled |>
  summarise(across(everything(), ~sum(is.na(.))))

na_counts <- na_counts |>
  select(where(~.x > 0))

print(na_counts)

# A tibble: 1 × 16
  HOME4NW2 BBHOME SMUSE_FB SMUSE_YT SMUSE_X SMUSE_IG SMUSE_SC SMUSE_WA SMUSE_TT
     <int>  <int>    <int>    <int>   <int>    <int>    <int>    <int>    <int>
1      176    720      176      176     176      176      176      176      176
# ℹ 7 more variables: SMUSE_RD <int>, SMUSE_BSK <int>, SMUSE_TH <int>,
#   SMUSE_TS <int>, SMART2 <int>, PARTYLN <int>, VOTEGEN_POST <int>


Most of those columns have a small number of missing values, representing 3-15% of the data in that column. However, PARTYLN, which asks for party leaning, is missing 63% of its inputs, and the data (party and leaning) is contained in the variables PARTY and PARTYSUM, so we will remove it.

VOTEGEN_POST is missign 20% of its data, and it asks responders to state who they voted for in the 2020 election.

After removing PARTYLN, we will change all NA values in the dataset to Refused/Web blank

In [10]:
npors_2025_labeled <- npors_2025_labeled |> select(-PARTYLN)

In [11]:
npors_2025_labeled <- npors_2025_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused/Web blank")))

In [12]:
na_counts <- npors_2025_labeled |>
  summarise(across(everything(), ~sum(is.na(.))))

na_counts <- na_counts |>
  select(where(~.x > 0))

print(na_counts)
#no na values left

# A tibble: 1 × 0


This dataset is clean, so we will save it, and move on to EDA with it

In [38]:
write_csv(npors_2025_labeled, '../data/cleandata/NPORS_2025_clean.csv')

That was NPORS 2025, here's the rest of them
## Other years

In [122]:
library(haven)

npors_2024 <- read_sav("../data/rawdata/NPORS-2024-Data-Release/NPORS_2024_for_public_release.sav")
npors_2023 <- read_sav("../data/rawdata/NPORS-2023-Data-Release 4/NPORS_2023_for_public_release.sav")
npors_2022 <- read_sav("../data/rawdata/NPORS_2022_for_public_release.sav")
npors_2021 <- read_sav("../data/rawdata/NPORS_2021_for_public_release.sav")
npors_2020 <- read_sav("../data/rawdata/NPORS 2020 Data Release-2-20260402T173758Z-3-001/NPORS 2020 Data Release-2/NPORS 2020.sav")

We don't have the codebook for previous years, but we can try to match what we already have, because some questions are the same

We also can get copies of the questions for each year, but that would be manual entry. 

In [123]:
common_cols <- Reduce(intersect, list(
  names(npors_2025),
  names(npors_2024),
  names(npors_2023),
  names(npors_2022),
  names(npors_2021),
  names(npors_2020)
))

common_cols

[1] "WEIGHT"

Ok so there is only one column that is used across the entire dataset. That kind of makes sense. We need to make a codebook based on the surveys

In [124]:
# codebooks made from surveys
codebook_2024 <- read.csv("../data/rawdata/codebooks/npors_2024_full_codebook.csv")
codebook_2023 <- read.csv("../data/rawdata/codebooks/npors_2023_value_codebook.csv")
codebook_2022 <- read.csv("../data/rawdata/codebooks/npors_2022_full_codebook.csv")
codebook_2021 <- read.csv("../data/rawdata/codebooks/npors_2021_full_codebook.csv")
codebook_2020 <- read.csv("../data/rawdata/codebooks/npors_2020_full_codebook.csv")

In [125]:
#making a lookup table
lookups_2024 <- codebook_2024 |>
  group_by(variable) |>
  summarise(map = list(setNames(label, value))) |>
  deframe()

#example
lookups_2024$ECON1MOD

lookups_2023 <- codebook_2023 |>
  group_by(variable) |>
  summarise(map = list(setNames(label, value))) |>
  deframe()

lookups_2022 <- codebook_2022 |>
  group_by(variable) |>
  summarise(map = list(setNames(label, value))) |>
  deframe()

lookups_2021 <- codebook_2021 |>
  group_by(variable) |>
  summarise(map = list(setNames(label, value))) |>
  deframe()

lookups_2020 <- codebook_2020 |>
  group_by(variable) |>
  summarise(map = list(setNames(label, value))) |>
  deframe()

In [128]:
#Some questions were asked in a straightforward mannar (1 number to 1 response), but others
# have multiple layers that were not uniformly communicated in the SVG. This code gets rid of
# the label mapping for those columns, they should be addressed on a case-by-case basis
#in reference to the survey questions 
lookups_2024[c("RACEMOD", "SMUSE", "AE1B")] <- NULL

lookups_2023[c("AE1B", "RACEMOD")] <- NULL
lookups_2022[c("AE1B", "RACEMOD", "CHR", "CITIZEN", "E1", "E2", "E3MOD", "EDUC_ACS", "IDEO", "INC_SDT1", "INC_SDT2", "KIDS1", "KIDS2", "LWP", "PARENT", "QA3", "SMUSE")] <- NULL
lookups_2021[c("AE1B_AP21", "AE2_AP21", "CHR_AP21", 
"CITIZEN_AP21", "COMTYPE2_AP21", "DEVICE1_AP21", 
"E1_AP21", "E2_AP21", "E3_AP21", "EDUC_ACS_AP21", "FOLGOV_AP21", "FOLLOWED_AP21", 
"HISPORIG1_AP21", "HISPORIG2_AP21","HLTHRATE_AP21","IDEO_AP21","INC_SDT1_AP21","INC_SDT2_AP21",
"KIDS1_AP21","KIDS2_AP21","LLB_AP21","LWP_AP21","OWNRENTMOD_AP21","PARENT_AP21","QA3_AP21",
"QA4_AP21","RACEMOD_AP21","RESPONSE_AP21","TYPOLOGY_AP21")] <- NULL
lookups_2020[c("AE1B_AP20", "AE2_AP20", "CHR_AP20", "CITIZEN_AP20", "COMTYPE2_AP20", "DEVICE1_AP20", "E1_AP20", "E2_AP20", "FOLGOV_AP20", "HISPORIG1_AP20", 
"HISPORIG2_AP20", "IDEO_AP20", "KIDS1_AP20", "KIDS2_AP20", "LLB_AP20", "LWP_AP20", "OWNRENTMOD_AP20",
"PARENT_AP20","PVOTE16A_AP20","PVOTE16B_AP20","QA3_AP20","COMTYPE2", "QA4_AP20", "RACEMOD_AP20","RESPONSE_AP20", "TYPOLOGY_AP20")] <- NULL


In [129]:
npors_2024_labeled <- npors_2024 |>
  mutate(across(
    names(lookups_2024),
    ~ factor(., levels = names(lookups_2024[[cur_column()]]),
             labels = lookups_2024[[cur_column()]])
  ))

In [130]:
npors_2023_labeled <- npors_2023 |>
  mutate(across(
    names(lookups_2023),
    ~ factor(., levels = names(lookups_2023[[cur_column()]]),
             labels = lookups_2023[[cur_column()]])
  ))

In [131]:
npors_2022_labeled <- npors_2022 |>
  mutate(across(
    names(lookups_2022),
    ~ factor(., levels = names(lookups_2022[[cur_column()]]),
             labels = lookups_2022[[cur_column()]])
  ))

In [132]:
npors_2021_labeled <- npors_2021 |>
  mutate(across(
    names(lookups_2021),
    ~ factor(., levels = names(lookups_2021[[cur_column()]]),
             labels = lookups_2021[[cur_column()]])
  ))

In [133]:
npors_2020_labeled <- npors_2020 |>
  mutate(across(
    names(lookups_2020),
    ~ factor(., levels = names(lookups_2020[[cur_column()]]),
             labels = lookups_2020[[cur_column()]])
  ))

Now we have to clean each of these a little bit more

In [134]:
# removing irrelevant or incomplete columns
npors_2024_labeled <- npors_2024_labeled |> select(-MODE, -LANGUAGE, -LANGUAGEINITIAL, -LANG_PREF, -SURVEYSTARTDATE,
-SURVEYENDDATE, -BWAVE, -PARTYLN, -AGE, -ADULTS, -VOTEGEN_POST)

npors_2023_labeled <- npors_2023_labeled |> select(-MODE_2WAY, -DATERECEIVED,-DEVICE_TYPE, -BOOKS1_REFUSED, -LANG_PREF, -INTERVIEW_START,
-INTERVIEW_END, -PARTYLN, -VOTEGEN_POST)

npors_2022_labeled <- npors_2022_labeled |> select(-MODE,-DEVICE_TYPE, -BOOKS1_REFUSED, -LANG_PREF, -INTERVIEW_START,
-INTERVIEW_END, -PARTYLN, -VOTEGEN_POST)

npors_2021_labeled <- npors_2021_labeled |> select(-MODE_AP21,-DATERECEIVED_AP21, -LANG_AP21, -INTERVIEW_START_AP21,
-INTERVIEW_END_AP21, -PARTYLN_AP21, -VOTEGEN_POST_AP21, -BORN_AP21)

npors_2020_labeled <- npors_2020_labeled |> select(-MODE, -LANGUAGE_AP20, -INTERVIEW_START_AP20,
-INTERVIEW_END_AP20, -PARTYLN_AP20, -INCOME_AP20)

In [ ]:
# turning NA values into missing

npors_2024_labeled <- npors_2024_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused/ Web blank")))

npors_2023_labeled <- npors_2023_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused")))

npors_2022_labeled <- npors_2022_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused")))

npors_2021_labeled <- npors_2021_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused")))

npors_2020_labeled <- npors_2020_labeled |>
  mutate(across(everything(), ~ replace_na(as.character(.), "Refused")))

In [136]:
#saving
write_csv(npors_2024_labeled, '../data/cleandata/NPORS_2024_clean.csv')
write_csv(npors_2023_labeled, '../data/cleandata/NPORS_2023_clean.csv')
write_csv(npors_2022_labeled, '../data/cleandata/NPORS_2022_clean.csv')
write_csv(npors_2021_labeled, '../data/cleandata/NPORS_2021_clean.csv')
write_csv(npors_2020_labeled, '../data/cleandata/NPORS_2020_clean.csv')